# Lab 2.2 — Tool / Function Calling on Mantle

**Duration:** ~10 min · **Level:** L300 · **Lab 2 of 4 — part 2/3**

Mantle exposes tool calling on all three surfaces, but with different
shapes. Migrating a Converse tool loop to Mantle is the single largest
source of code changes in a real migration.

Scope of this notebook:

1. Define a tiny `get_weather` tool and call it via **Chat Completions**.
2. Show the critical `json.loads()` step for tool arguments on the OpenAI
   surfaces.
3. Run the same tool via the **Anthropic Messages** path (no `json.loads()`
   needed — arguments arrive parsed).
4. Preview **server-side tool execution** on the Responses API (MCP /
   Lambda).


In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "src" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

from src.common.mantle import (
    openai_client, anthropic_client,
    GPT_OSS_120B, CLAUDE_OPUS_47, CLAUDE_HAIKU_45,
)

client = openai_client()
anth = anthropic_client()

# The "tool" is a dictionary lookup. In real code it would call an API.
_WEATHER = {"paris": {"temp_c": 22, "condition": "sunny"},
            "seattle": {"temp_c": 15, "condition": "drizzle"},
            "tokyo": {"temp_c": 27, "condition": "humid"}}

def get_weather(city: str) -> dict:
    return _WEATHER.get(city.lower(), {"error": f"no data for {city!r}"})


## 1. Chat Completions — full tool loop

**Schema wrapper:** `tools[]` with a `function` envelope, `parameters` is a
standard JSON schema.

**`tool_choice`:** `"auto"` / `"required"` / `{"type":"function","function":{"name":"…"}}`.

**Arguments on the wire:** `tool_calls[].function.arguments` is a **JSON
string**, not a dict. You must `json.loads()` it before using it.

**Tool result turn-in:** a dedicated `role: "tool"` message carrying
`tool_call_id` and a **string** `content` (JSON-encode your result).


In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
            "additionalProperties": False,
        },
    },
}]

messages = [
    {"role": "system", "content": "Use the get_weather tool whenever the user asks about weather."},
    {"role": "user",   "content": "Compare the weather in Paris and Tokyo."},
]

MAX_ITER = 6    # cap so a misbehaving model can't infinite-loop us.
for iteration in range(MAX_ITER):
    r = client.chat.completions.create(
        model=GPT_OSS_120B,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        max_tokens=400,
    )
    msg = r.choices[0].message

    # Preserve the assistant turn verbatim. Content may be None while
    # tool_calls are being requested — that's expected on OpenAI.
    assistant_turn = {"role": "assistant", "content": msg.content}
    if msg.tool_calls:
        assistant_turn["tool_calls"] = [tc.model_dump() for tc in msg.tool_calls]
    messages.append(assistant_turn)

    if r.choices[0].finish_reason != "tool_calls":
        print("--- final answer ---")
        print(msg.content)
        break

    # Execute each tool call the model requested.
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)   # STRING → dict
        print(f"[tool] {tc.function.name}({args})")
        result = get_weather(**args)
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(result),          # STRING content
        })
else:
    print(f"⚠️  loop hit MAX_ITER={MAX_ITER} without terminating")


### Watch the loop terminate

The while-loop exits when `finish_reason == "stop"`. Other terminals you
should handle in production:

- `length` — ran out of `max_tokens` mid-tool-call. Raise `max_tokens`.
- `content_filter` — Bedrock guardrail or model refusal.
- `tool_calls` persisting — the model is convinced it needs another round;
  add an iteration cap so you can't infinite-loop on bad prompts.


## 2. Anthropic Messages — same tool, native shape

On the Anthropic surface:

- **Schema wrapper:** `tools[].input_schema` (no `function` envelope).
- **`tool_choice`:** `{"type":"auto"}` / `{"type":"any"}` /
  `{"type":"tool","name":"..."}` — *not* the bare strings used on OpenAI.
- **Arguments:** arrive as a parsed **dict** under `content[].input` — no
  `json.loads()`.
- **Tool result turn-in:** a `user` message with `tool_result` content
  blocks, *not* a `tool` role.


In [ ]:
anth_tools = [{
    "name": "get_weather",
    "description": "Current weather for a city",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]

anth_messages = [
    {"role": "user", "content": "Compare the weather in Paris and Seattle."}
]

MAX_ITER = 6
for iteration in range(MAX_ITER):
    r = anth.messages.create(
        model=CLAUDE_HAIKU_45,
        max_tokens=400,
        system="Use the get_weather tool for any weather question.",
        tools=anth_tools,
        messages=anth_messages,
    )

    # Preserve the assistant turn (mix of text and tool_use blocks).
    anth_messages.append({"role": "assistant", "content": r.content})

    if r.stop_reason != "tool_use":
        text = next((b.text for b in r.content if b.type == "text"), "")
        print("--- final answer ---")
        print(text)
        break

    tool_results = []
    for block in r.content:
        if block.type != "tool_use":
            continue
        # .input is ALREADY a dict here — no json.loads()
        print(f"[tool] {block.name}({block.input})")
        out = get_weather(**block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": json.dumps(out),            # strings are lingua franca
        })
    anth_messages.append({"role": "user", "content": tool_results})
else:
    print(f"⚠️  loop hit MAX_ITER={MAX_ITER} without terminating")


### Axis-A vs Axis-B differences to memorise

From the playbook:

- **Axis A** — *Converse (runtime) → any Mantle surface.* The schema
  wrapper, stop-reason vocab, and message-history shape all change.
- **Axis B** — *between Mantle surfaces.* Chat Completions has strings and
  a dedicated `tool` role; Anthropic has dicts and `user`+`tool_result`
  blocks. Your provider abstraction must branch *three* ways, not one.

| Shape | Chat Completions | Responses API | Anthropic Messages |
|---|---|---|---|
| Tool schema wrapper | `tools[].function.parameters` | `tools[]` flat | `tools[].input_schema` |
| `tool_choice=auto` | `"auto"` | `"auto"` | `{"type":"auto"}` |
| `tool_choice=required` | `"required"` | `"required"` | `{"type":"any"}` |
| Arguments on wire | **JSON string** | **JSON string** (on `.done` event) | **dict** |
| Tool result turn-in | `role:"tool"` + `tool_call_id` | `function_call_output` item | `user` + `tool_result` block |
| Rich tool results (image / JSON) | strings only | strings only | typed blocks |
| Stop-reason keyword | `finish_reason == "tool_calls"` | walk `output[]` | `stop_reason == "tool_use"` |


## 3. Server-side tools (preview — Responses API only)

The Responses API lets Mantle run the tool loop *inside Bedrock*. You
register an MCP-compatible Lambda and the model calls it directly — no
round-trip to your client between tool calls.

The snippet below is for reference; running it requires you to deploy a
Lambda with the MCP contract (`tools/list`, `tools/call`) first. Lab 4
shows the end-to-end wiring.


In [ ]:
# NOTE: This cell is illustrative. Uncomment and replace <account> +
# <fn-name> after you deploy an MCP-compatible Lambda in your account.
#
# response = client.responses.create(
#     model=GPT_OSS_120B,
#     tools=[{
#         "type": "mcp",
#         "server_label": "product_tools",
#         "connector_id": "arn:aws:lambda:us-east-1:<account>:function:<fn-name>",
#         "require_approval": "never",
#     }],
#     input="Find laptops under $1000 and list their prices.",
# )
# print(response.output_text)
print("server-side tool snippet shown as a comment — no live call is made in this cell.")


## 4. Migration checklist

When you port a Converse tool loop to Mantle Chat Completions:

- [ ] Flatten `toolSpec.inputSchema.json` → `function.parameters`.
- [ ] Remap `toolChoice`: `{"any":{}}` → `"required"`,
      `{"tool":{"name":"x"}}` → `{"type":"function","function":{"name":"x"}}`.
- [ ] Switch `stopReason == "tool_use"` → `finish_reason == "tool_calls"`.
- [ ] **`json.loads()` every tool argument.** They arrive as strings.
- [ ] Emit `role:"tool"` with `tool_call_id` (not `user` + `toolResult`).
- [ ] Collapse image / document / JSON tool results to strings
      (JSON-encode dicts, stringify numbers).
- [ ] Set `parallel_tool_calls` explicitly (defaults vary per model).
- [ ] Cap loop iterations (`for _ in range(8)`) to prevent runaways.
- [ ] Add telemetry for `(tool_name, args, latency, ok)` — Mantle itself
      does not log tool names by default.
